# SE2_data_processing

**Thesis:** Near-Real-Time Streamflow and Nutrient Prediction for the Oulanka River Using SWAT+ and Open Meteorological APIs  
**Author:** Masters Student, Water, Energy and Environmental Engineering (WE3), University of Oulu  

---

## Overview

This notebook is **Step 2** of the four-part reproducible pipeline (SE1 → SE4).  
It reads `df_obs` and `df_forecast` produced by SE1 and converts them into **analysis-ready SWAT+ climate input files**.

### Input data description
| Variable | Source | Temporal resolution | Spatial resolution | CRS |
|----------|--------|--------------------|--------------------|-----|
| `df_obs` | SE1 (FMI point / FMI grid / ERA5) | Daily | Point or nearest grid cell | N/A |
| `df_forecast` | SE1 Harmonie NWP (Option 1 only) | Daily aggregated from hourly | Point 66.364°N, 29.316°E | N/A |
| `TxtInOut_test/` | Zenodo (SE1 Part A) | — | Oulanka catchment | ETRS-TM35FIN |

### Outputs
Two parallel TxtInOut folders are created outside the repository:

| Folder | Contents | Purpose |
|--------|----------|--------|
| `TxtInOut_Obs_Only_YYYY-MM-DD` | Historical observations only, no forecast | Safe archive — grows incrementally, never overwritten |
| `TxtInOut_Full_Execution_YYYY-MM-DD` | Observations + Harmonie forecast | Ready for SWAT+ run in SE3 — recreated fresh every run |

### Modelling framework
SWAT+ v61.0.2 via `pySWATPlus`. Climate files must follow strict Fortran fixed-width format.  
SE3 reads directly from `TxtInOut_Full_Execution_YYYY-MM-DD`.

### Data processing steps
1. Read station file list from `weather-sta.cli`
2. Create Folder A (observation archive) from the most recent existing snapshot
3. Append `df_obs` to Folder A and Folder B in Fortran fixed-width format
4. Create Folder B (execution copy) from Folder A
5. Append `df_forecast` to Folder B only
6. Update `nbyr` header in `weather-sta.cli` for both folders
7. Verify output with a tail-print of the final `.pcp` file

### Storage requirements
Each TxtInOut folder: approximately 50–150 MB.  
Two folders per run: approximately 100–300 MB additional storage.

In [ ]:
!df -k ..
# Confirm sufficient storage is available before creating TxtInOut copies.

## Cell 1 — Load dependencies

In [ ]:
# Modify cell
from pathlib import Path
import os
import glob
import shutil
import datetime
import pandas as pd
import numpy as np

print("Dependencies loaded successfully.")

## Cell 2 — Directory setup

All processed data lives **outside** the repository so large TxtInOut folders are never committed to GitHub.  
Only `BASE_DIR` needs updating if you move the project to a new machine.

In [ ]:
# Automatically generated cell — resolve external directories
# Current working directory = project/notebooks
notebook_dir  = Path.cwd()
project_root  = notebook_dir.parent
external_base = project_root.parent

# Processed data directory outside the repository
proc_data_dir = external_base / "oulanka_swatplus_processeddata"
proc_data_dir.mkdir(parents=True, exist_ok=True)

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
# This should match ZENODO_DIR / "TxtInOut_test" from SE1
BASE_DIR     = str(external_base / "oulanka_swatplus_rawdata" / "zenodo_data")  # <-- update if needed
TXTINOUT_DIR = os.path.join(BASE_DIR, "TxtInOut_test")   # base model folder from Zenodo

# Today's dated folder names (one pair created per calendar day)
today_str = datetime.datetime.now().strftime('%Y-%m-%d')
folder_a  = os.path.join(BASE_DIR, f"TxtInOut_Obs_Only_{today_str}")
folder_b  = os.path.join(BASE_DIR, f"TxtInOut_Full_Execution_{today_str}")

print("Processed data directory :", proc_data_dir.resolve())
print("Base SWAT+ directory     :", BASE_DIR)
print("Observation archive (A)  :", folder_a)
print("Execution folder    (B)  :", folder_b)

## Cell 3 — Fortran formatting helpers

SWAT+ reads climate files in strict Fortran fixed-width format.  
Each data line must follow: `YYYY  JJJ VVVVVVVVVVV`  
(4-char year, 2 spaces, 3-char Julian day, 11-char value with 5 decimal places)

Two data quality cleaners handle FMI-specific issues:
- `clean_pcp`: FMI reports trace precipitation as −1.0 → converted to 0.0  
- `clean_val`: General NaN-safe float converter for temperature, humidity, radiation

A third helper reads the last date already written in a SWAT+ climate file,
enabling safe incremental appending without duplicating existing data.

In [ ]:
# ==========================================
# CELL 3: FORTRAN FORMATTING HELPERS
# ==========================================

def clean_pcp(v):
    """
    Clean precipitation value for SWAT+ input.
    FMI uses -1.0 for trace amounts; negatives and NaNs are set to 0.0.
    """
    if isinstance(v, dict):
        v = v.get('value', 0.0)
    try:
        val = float(v)
        return 0.0 if (pd.isna(val) or val < 0) else val
    except:
        return 0.0


def clean_val(v):
    """
    General cleaner for temperature, humidity, and solar radiation.
    NaNs are replaced with 0.0.
    """
    if isinstance(v, dict):
        v = v.get('value', 0.0)
    try:
        return float(v) if not pd.isna(float(v)) else 0.0
    except:
        return 0.0


def format_swat_1val(year, julian_day, value, is_pcp=False):
    """
    Format a single-value SWAT+ climate file line.
    Used for: .pcp, .slr, .hmd files.
    Format: 'YYYY  JJJ VVVVVVVVVVV' (11-char value, 5 decimal places)
    """
    val = clean_pcp(value) if is_pcp else clean_val(value)
    return f"{year:4d}  {julian_day:3d}{val:11.5f}\n"


def format_swat_2val(year, julian_day, tmax, tmin):
    """
    Format a two-value SWAT+ climate file line.
    Used for: .tmp files (Tmax and Tmin on the same line).
    Format: 'YYYY  JJJ TTTTTTTTTTT TTTTTTTTTTT'
    """
    return f"{year:4d}  {julian_day:3d}{clean_val(tmax):11.5f}{clean_val(tmin):11.5f}\n"


def get_last_date_in_file(filepath):
    """
    Read the last Julian date from an existing SWAT+ climate file.
    Returns a datetime object or None if the file is empty / has header only.
    Used to avoid appending duplicate data on repeated runs.
    """
    if not os.path.exists(filepath):
        return None
    with open(filepath, 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    if len(lines) < 3:
        return None
    try:
        parts = lines[-1].split()
        yr    = int(parts[0])
        doy   = int(parts[1])
        return datetime.datetime(yr, 1, 1) + datetime.timedelta(doy - 1)
    except:
        return None


print("Formatting helper functions defined.")

## Cell 4 — Create Folder A (observation archive) and Folder B (execution copy)

**Folder A** is a safe, forecast-free archive. It is created once per calendar day and never overwritten.  
**Folder B** is always recreated fresh from Folder A before each prediction run.

The source for copying is always the most recent existing `TxtInOut_Obs_Only_*` snapshot on disk.  
If none exists yet (first ever run), the base `TxtInOut_test` folder from Zenodo is used as the starting point.

In [ ]:
# ==========================================
# CELL 4: CREATE DUAL FOLDERS (A & B)
# ==========================================

# --- 1. Find the most recent valid observation archive as copy source ---
obs_archives   = sorted(glob.glob(os.path.join(BASE_DIR, "TxtInOut_Obs_Only_*")))
valid_archives = [f for f in obs_archives if f != folder_a and os.path.exists(f)]
source_folder  = valid_archives[-1] if valid_archives else TXTINOUT_DIR

print(f"Copy source: {os.path.basename(source_folder)}")

# --- 2. Create Folder A (idempotent: only created once per calendar day) ---
if not os.path.exists(folder_a):
    shutil.copytree(source_folder, folder_a)
    print(f"Created observation archive : {os.path.basename(folder_a)}")
else:
    print(f"Archive already exists today: {os.path.basename(folder_a)}")

# --- 3. Create Folder B (always recreated fresh from Folder A) ---
if os.path.exists(folder_b):
    shutil.rmtree(folder_b)
shutil.copytree(folder_a, folder_b)
print(f"Created execution folder   : {os.path.basename(folder_b)}")

## Cell 5 — Read station list from weather-sta.cli

`weather-sta.cli` defines which `.pcp`, `.tmp`, `.slr`, and `.hmd` files belong to which station.  
Reading it dynamically means this code works for any SWAT+ model regardless of how many stations it has — whether it is a single FMI point station or multiple grid stations.

In [ ]:
# ==========================================
# CELL 5: READ STATION LIST FROM weather-sta.cli
# ==========================================

cli_path = os.path.join(folder_a, "weather-sta.cli")

with open(cli_path, 'r') as f:
    raw_lines = f.readlines()

# Lines 0 and 1 are header; data starts from line 2
# weather-sta.cli column order: name  lat  lon  pcp  tmp  slr  hmd  wnd
stations = [
    line.split()
    for line in raw_lines[2:]
    if len(line.split()) >= 6
]

print(f"Found {len(stations)} station(s) in weather-sta.cli:")
for st in stations:
    print(f"  name={st[0]}  pcp={st[2]}  tmp={st[3]}  slr={st[4]}  hmd={st[5]}")

## Cell 6 — Append historical observations (df_obs) to Folder A and Folder B

`df_obs` from SE1 contains only the **new days** not yet present in the SWAT+ files.  
Each station's climate file is appended; `get_last_date_in_file()` is used as a double-check
to skip any dates already written, making this step safe to re-run without duplication.

If `df_obs` is empty (data already up to date), this cell prints a message and moves on — no error.

In [ ]:
# ==========================================
# CELL 6: APPEND OBSERVATIONS TO FOLDERS A & B
# ==========================================

if 'df_obs' in globals() and not df_obs.empty:

    for target_folder in [folder_a, folder_b]:
        print(f"\nAppending {len(df_obs)} observation days to: {os.path.basename(target_folder)}")

        for st in stations:

            # ── 1-value files: precipitation, solar radiation, humidity ──────────
            # weather-sta.cli order: name lat lon pcp tmp slr hmd
            for key, fname in [('pcp', st[2]), ('slr', st[4]), ('hmd', st[5])]:
                if fname == 'null':
                    continue
                fpath    = os.path.join(target_folder, fname)
                last_dt  = get_last_date_in_file(fpath)   # avoid duplicate lines
                new_rows = df_obs if last_dt is None else df_obs[df_obs.index > last_dt]

                with open(fpath, 'a') as f:
                    for dt, row in new_rows.iterrows():
                        is_pcp = (key == 'pcp')
                        f.write(format_swat_1val(dt.year, dt.timetuple().tm_yday, row[key], is_pcp))

            # ── 2-value file: temperature (Tmax and Tmin on same line) ───────────
            t_fname  = st[3]
            fpath_t  = os.path.join(target_folder, t_fname)
            last_dt  = get_last_date_in_file(fpath_t)
            new_rows = df_obs if last_dt is None else df_obs[df_obs.index > last_dt]

            with open(fpath_t, 'a') as f:
                for dt, row in new_rows.iterrows():
                    f.write(format_swat_2val(dt.year, dt.timetuple().tm_yday, row['tmax'], row['tmin']))

    print("\nObservation data appended successfully. Fortran spacing verified.")

else:
    print("df_obs is empty — no new observation days to append. Folders are already up to date.")

## Cell 7 — Append Harmonie forecast (df_forecast) to Folder B only

`df_forecast` is appended **only to Folder B**. Folder A stays as a clean observation-only archive.

If `df_forecast` is empty — which happens when `FORCING_OPTION` is `"FMI_GRID"` or `"ERA5"` in SE1 —
this cell skips silently. Folder B is still valid and usable; SE3 will simply simulate up to the
last available observation day rather than extending into the future.

In [ ]:
# ==========================================
# CELL 7: APPEND FORECAST TO FOLDER B ONLY
# ==========================================

if 'df_forecast' in globals() and not df_forecast.empty:

    print(f"Appending {len(df_forecast)} forecast days to: {os.path.basename(folder_b)}")

    for st in stations:

        # 1-value files: precipitation, solar radiation, humidity
        for key, fname in [('pcp', st[2]), ('slr', st[4]), ('hmd', st[5])]:
            if fname == 'null':
                continue
            with open(os.path.join(folder_b, fname), 'a') as f:
                for dt, row in df_forecast.iterrows():
                    is_pcp = (key == 'pcp')
                    f.write(format_swat_1val(dt.year, dt.timetuple().tm_yday, row[key], is_pcp))

        # 2-value file: temperature
        with open(os.path.join(folder_b, st[3]), 'a') as f:
            for dt, row in df_forecast.iterrows():
                f.write(format_swat_2val(dt.year, dt.timetuple().tm_yday, row['tmax'], row['tmin']))

    print("Harmonie forecast appended to Folder B successfully.")

else:
    print("df_forecast is empty (FMI_GRID or ERA5 forcing option, or Harmonie unavailable).")
    print("Folder B contains observations only. SE3 will simulate up to the last observation day.")

## Cell 8 — Update nbyr header in weather-sta.cli

`nbyr` (number of years) in `weather-sta.cli` tells SWAT+ how many years of data the climate files contain.  
If this value is wrong, SWAT+ will either stop reading too early or crash on missing data.  
Both Folder A and Folder B are updated.

In [ ]:
# ==========================================
# CELL 8: UPDATE nbyr IN weather-sta.cli
# ==========================================

current_year = datetime.datetime.now().year

# SWAT+ data in this project starts from 1990 (update BASE_YEAR if your data starts earlier)
BASE_YEAR = 1990
nbyr      = current_year - BASE_YEAR + 1

for target_folder in [folder_a, folder_b]:
    cli_update_path = os.path.join(target_folder, "weather-sta.cli")
    with open(cli_update_path, 'r') as f:
        lines = f.readlines()
    # Line index 1 contains the nbyr value
    lines[1] = f"{nbyr:8d}    nbyr: number of years\n"
    with open(cli_update_path, 'w') as f:
        f.writelines(lines)
    print(f"Updated nbyr = {nbyr} in: {os.path.basename(target_folder)}")

## Cell 9 — Verification: print tail of the final .pcp file

A quick sanity check: print the last 10 lines of the precipitation file in Folder B.  
This confirms that:
- Fortran column alignment is correct (year, Julian day, value)
- The dates are sequential with no gaps
- The last date matches what is expected (yesterday for obs, or forecast end date)

In [ ]:
# ==========================================
# CELL 9: VERIFICATION — TAIL OF .pcp FILE
# ==========================================

first_pcp_fname = stations[0][2]   # first station's precipitation file name
first_pcp_path  = os.path.join(folder_b, first_pcp_fname)

print(f"Last 10 lines of: {first_pcp_fname}  (Folder B)")
print("-" * 45)

with open(first_pcp_path, 'r') as f:
    all_lines = f.readlines()

for line in all_lines[-10:]:
    print(line, end='')

print("-" * 45)
print(f"Total lines in file : {len(all_lines)}")
print(f"\nFolder A (obs only) : {os.path.basename(folder_a)}")
print(f"Folder B (exec ready): {os.path.basename(folder_b)}")
print("\nSE2 complete. SE3 can now run SWAT+ using Folder B.")

## Summary of outputs

| Folder | Contents | Safe to keep? |
|--------|----------|---------------|
| `TxtInOut_Obs_Only_YYYY-MM-DD` | Observations only, no forecast | Yes — archive it; it grows incrementally |
| `TxtInOut_Full_Execution_YYYY-MM-DD` | Observations + forecast | Overwritten each run — do not rely on it across days |

SE3 reads from `TxtInOut_Full_Execution_YYYY-MM-DD` via the `EXEC_FOLDER` variable.

---
## Need help?
Search, ask, and answer data processing questions at https://github.com/orgs/DigitalWaters-fi/discussions  
Tag **#dataprocess**